# QamomileのPeriodic Stencil Block Encodingで解く周期Poisson方程式

このノートブックはPR #549のQSVT Poissonノートブックを、今回追加した
periodic_stencil_block_encodingで書き換えた日本語版です。

元のノートブックでは、1次元用と2次元用にPREPARE、制御付き周期shift、
UNPREPAREをそれぞれ手書きしていました。ここでは次の2行でencodingを作り、
1次元・2次元に共通なQSVT factoryから利用します。

    encoding = qmc.periodic_stencil_block_encoding(coefficients, register_sizes)
    signal, system = encoding.unitary(signal, system)

例はexact OperatorとStatevectorで確認するため、小さい格子に限定します。
大きい問題ではexact simulationではなくsamplingやresource estimationを
利用してください。

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from qiskit.quantum_info import Operator, Statevector

import qamomile.circuit as qmc
from qamomile.qiskit import QiskitTranspiler

## 1. 周期Poisson方程式とshift表現

1次元の周期離散Laplacianを

$$
(Au)_j = \frac{u_{j+1}-2u_j+u_{j-1}}{h^2}
$$

とします。周期shiftを $T_{+1}|j\rangle=|j+1\bmod N\rangle$、
$T_{-1}=T_{+1}^{\dagger}$ と書けば、

$$
A_{\mathrm{1D}}
= \frac{1}{h^2}\left(-2I+T_{-1}+T_{+1}\right)
$$

です。2次元では5点stencil

$$
A_{\mathrm{2D}}
= \frac{1}{h^2}\left(
-4I+T_{(-1,0)}+T_{(1,0)}+T_{(0,-1)}+T_{(0,1)}
\right)
$$

を使います。

Periodic Stencil factoryは

$$
A=\sum_k c_kT_k,\qquad
\lambda=\sum_k|c_k|
$$

からunitary $U$を作り、signal registerを全て0へ射影したblockが

$$
(\langle0|\otimes I)U(|0\rangle\otimes I)=A/\lambda
$$

となるようにします。ここでlambdaはencoding.normalizationから取得できます。

In [ ]:
def check_power_of_two(value):
    """Validate that a positive integer is a power of two."""
    if value <= 0 or value & (value - 1):
        raise ValueError("value must be a positive power of two.")


def normalize_state(vector):
    """Return a normalized real vector for amplitude encoding."""
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)
    if norm == 0.0:
        raise ValueError("The state vector must not be zero.")
    return vector / norm


def project_periodic_rhs(vector):
    """Project a periodic Poisson source onto the Laplacian range."""
    vector = np.asarray(vector, dtype=float)
    return vector - np.mean(vector)


def fidelity(left, right):
    """Return the squared overlap of two nonzero vectors."""
    left = normalize_state(left)
    right = normalize_state(right)
    return float(abs(np.vdot(left, right)) ** 2)


def build_1d_periodic_problem(num_points, length=1.0, source=None):
    """Build a 1D periodic finite-difference Poisson problem."""
    check_power_of_two(num_points)
    spacing = length / num_points
    grid = np.arange(num_points) * spacing
    matrix = np.zeros((num_points, num_points), dtype=float)
    for row in range(num_points):
        matrix[row, row] = -2.0
        matrix[row, (row - 1) % num_points] = 1.0
        matrix[row, (row + 1) % num_points] = 1.0
    matrix /= spacing**2
    if source is None:
        rhs = np.zeros(num_points, dtype=float)
    else:
        rhs = np.asarray([source(point) for point in grid], dtype=float)
    return matrix, rhs, grid, spacing


def build_2d_periodic_problem(num_points, length=1.0, source=None):
    """Build a 2D periodic five-point Poisson problem."""
    matrix_1d, _, grid, spacing = build_1d_periodic_problem(
        num_points,
        length=length,
    )
    identity = np.eye(num_points)
    matrix = np.kron(matrix_1d, identity) + np.kron(identity, matrix_1d)
    grid_x, grid_y = np.meshgrid(grid, grid)
    if source is None:
        rhs = np.zeros(num_points * num_points, dtype=float)
    else:
        rhs = np.asarray(source(grid_x, grid_y), dtype=float).reshape(-1)
    return matrix, rhs, grid_x, grid_y, spacing


def recover_scaled_solution(matrix, rhs, postselected_state):
    """Scale a postselected QSVT vector by least squares."""
    candidate = np.real(np.asarray(postselected_state, dtype=complex))
    transformed = matrix @ candidate
    denominator = float(np.vdot(transformed, transformed).real)
    if denominator <= 1e-14:
        raise ValueError("The postselected QSVT component is numerically zero.")
    scale = float(np.vdot(transformed, rhs).real / denominator)
    return scale * candidate


transpiler = QiskitTranspiler()

## 2. 1次元encodingをfactoryで作る

register_sizesには格子点数ではなく、各axis registerのqubit数を渡します。
8点格子ならsystemは3 qubitです。係数には物理的な $1/h^2$ も含めるため、
encoding.normalizationは $4/h^2$ になります。

In [ ]:
num_points_1d = 8
matrix_1d, raw_rhs_1d, grid_1d, spacing_1d = build_1d_periodic_problem(
    num_points_1d,
    source=lambda x: np.sin(2.0 * np.pi * x),
)
rhs_1d = project_periodic_rhs(raw_rhs_1d)
reference_1d = np.linalg.pinv(matrix_1d) @ rhs_1d
normalized_rhs_1d = normalize_state(rhs_1d)
num_system_1d = int(np.log2(num_points_1d))

encoding_1d = qmc.periodic_stencil_block_encoding(
    {
        -1: 1.0 / spacing_1d**2,
        0: -2.0 / spacing_1d**2,
        1: 1.0 / spacing_1d**2,
    },
    register_sizes=(num_system_1d,),
)

print("1D normalization:", encoding_1d.normalization)
print("1D signal qubits:", encoding_1d.num_signal_qubits)
print("1D system qubits:", encoding_1d.num_system_qubits)
print("1D canonical offsets:", encoding_1d.offsets)

descriptorは回路そのものではありません。回路として呼ぶ対象はunitaryです。
systemとsignalを別々のregisterとして確保すると、通常実行、inverse、
control、nested SELECTで同じABIを利用できます。

In [ ]:
def make_block_encoding_check_kernel(encoding):
    """Build a measured kernel for exact leading-block inspection."""

    @qmc.qkernel
    def block_encoding_check() -> tuple[
        qmc.Vector[qmc.Bit],
        qmc.Vector[qmc.Bit],
    ]:
        # Allocate system first so it occupies the least-significant wires.
        system = qmc.qubit_array(encoding.num_system_qubits, "system")
        signal = qmc.qubit_array(encoding.num_signal_qubits, "signal")
        signal, system = encoding.unitary(signal, system)
        return qmc.measure(system), qmc.measure(signal)

    return block_encoding_check


def exact_operator(kernel, bindings=None):
    """Return the exact unitary emitted by a measured Qamomile kernel."""
    circuit = transpiler.to_circuit(kernel, bindings=bindings)
    circuit = circuit.remove_final_measurements(inplace=False)
    return Operator(circuit).data


def extract_encoded_matrix(encoding):
    """Extract the all-zero signal block from an encoding unitary."""
    kernel = make_block_encoding_check_kernel(encoding)
    full_operator = exact_operator(kernel)
    dimension = 1 << encoding.num_system_qubits
    leading_block = full_operator[:dimension, :dimension]
    return encoding.normalization * leading_block


encoded_matrix_1d = extract_encoded_matrix(encoding_1d)
error_1d = np.max(np.abs(encoded_matrix_1d - matrix_1d))
print("1D block matches the finite-difference matrix:", error_1d < 1e-8)
print("1D maximum absolute error:", error_1d)
assert np.allclose(encoded_matrix_1d, matrix_1d, atol=1e-8)

## 3. source stateのMöttönen amplitude encoding

source vectorはQamomileのamplitude_encodingでsystem registerへ読み込みます。
ここでもexact statevectorで準備結果を確認します。

In [ ]:
def make_state_preparation_kernel(amplitudes):
    """Build a measured amplitude-preparation kernel."""
    amplitudes = normalize_state(amplitudes)
    num_qubits = int(np.log2(len(amplitudes)))

    @qmc.qkernel
    def state_preparation() -> qmc.Vector[qmc.Bit]:
        system = qmc.qubit_array(num_qubits, "system")
        system = qmc.amplitude_encoding(system, amplitudes)
        return qmc.measure(system)

    return state_preparation


preparation_1d = make_state_preparation_kernel(normalized_rhs_1d)
preparation_circuit_1d = transpiler.to_circuit(
    preparation_1d
).remove_final_measurements(inplace=False)
prepared_1d = Statevector.from_instruction(preparation_circuit_1d).data
phase = np.vdot(prepared_1d, normalized_rhs_1d)
if abs(phase) > 1e-14:
    prepared_1d *= phase / abs(phase)
print(
    "source preparation matches:",
    np.allclose(prepared_1d, normalized_rhs_1d, atol=1e-8),
)

## 4. signal projector phase

QSVTではblock encodingのunitary $U$ と $U^\dagger$ の間に、
signalのall-zero subspaceへ作用するprojector phase

$$
R_\Pi(\phi)=\exp\left(i\phi(2\Pi-I)\right),
\qquad \Pi=|0\cdots0\rangle\langle0\cdots0|
$$

を挟みます。補助qubitはQSVT consumerが所有するworkspaceであり、
encoding.num_signal_qubitsには含まれません。各projector phase内で
compute/uncomputeされます。

元ノートブックの1本のflat registerをsliceする構成では、unitary呼び出し後に
親registerへ触れるとborrow conflictになります。ここではsystem、signal、
projector_auxを独立して確保します。

In [ ]:
def make_projector_phase(num_signal_qubits):
    """Build the all-zero-signal projector phase for a fixed width."""
    if num_signal_qubits <= 0:
        raise ValueError("num_signal_qubits must be positive.")

    if num_signal_qubits == 1:
        controlled_z = qmc.control(qmc.z, num_controls=1)

        @qmc.qkernel
        def controlled_minus_zpi(
            aux: qmc.Qubit,
            signal: qmc.Vector[qmc.Qubit],
        ) -> tuple[qmc.Qubit, qmc.Vector[qmc.Qubit]]:
            signal = qmc.x(signal)
            aux, signal[0] = controlled_z(aux, signal[0])
            signal = qmc.x(signal)
            return aux, signal

    else:
        controlled_z = qmc.control(qmc.z, num_controls=num_signal_qubits)

        @qmc.qkernel
        def controlled_minus_zpi(
            aux: qmc.Qubit,
            signal: qmc.Vector[qmc.Qubit],
        ) -> tuple[qmc.Qubit, qmc.Vector[qmc.Qubit]]:
            signal = qmc.x(signal)
            (
                aux,
                signal[0 : num_signal_qubits - 1],
                signal[num_signal_qubits - 1],
            ) = controlled_z(
                aux,
                signal[0 : num_signal_qubits - 1],
                signal[num_signal_qubits - 1],
            )
            signal = qmc.x(signal)
            return aux, signal

    @qmc.qkernel
    def projector_phase(
        aux: qmc.Qubit,
        signal: qmc.Vector[qmc.Qubit],
        two_phi: qmc.Float,
    ) -> tuple[qmc.Qubit, qmc.Vector[qmc.Qubit]]:
        # Compute C_Pi NOT.
        aux = qmc.h(aux)
        aux, signal = controlled_minus_zpi(aux, signal)
        aux = qmc.h(aux)

        # RZ(2 phi) = exp(-i phi Z).
        aux = qmc.rz(aux, two_phi)

        # Uncompute C_Pi NOT.
        aux = qmc.h(aux)
        aux, signal = controlled_minus_zpi(aux, signal)
        aux = qmc.h(aux)
        return aux, signal

    return projector_phase

## 5. encoding共通契約を使うQSVT factory

このfactoryは1次元か2次元かを知りません。必要なのは

- encoding.unitary
- encoding.num_signal_qubits
- encoding.num_system_qubits

だけです。これが今回そろえた共通契約の具体的な利点です。

位相を2個ずつ消費する部分は

$$
R_\Pi(\phi_{2j})\;U\;
R_\Pi(\phi_{2j+1})\;U^\dagger
$$

です。位相数が偶数なら最後に
$R_\Pi(\phi)UR_\Pi(\phi')$、奇数なら最後に
$R_\Pi(\phi)$ を置きます。

In [ ]:
def qsvt_num_pairs(num_phases):
    """Return the number of complete U/U-dagger phase pairs."""
    if num_phases <= 0:
        raise ValueError("num_phases must be positive.")
    if num_phases % 2 == 0:
        return (num_phases - 2) // 2
    return (num_phases - 1) // 2


def make_qsvt_poisson_kernel(encoding, normalized_rhs, num_phases):
    """Build a QSVT kernel for any compatible static block encoding."""
    normalized_rhs = normalize_state(normalized_rhs)
    expected_dimension = 1 << encoding.num_system_qubits
    if len(normalized_rhs) != expected_dimension:
        raise ValueError("The source length must match the encoding system dimension.")
    if num_phases <= 0:
        raise ValueError("num_phases must be positive.")

    projector_phase = make_projector_phase(encoding.num_signal_qubits)
    inverse_unitary = qmc.inverse(encoding.unitary)

    if num_phases % 2 == 0:

        @qmc.qkernel
        def qsvt_kernel(
            phases: qmc.Vector[qmc.Float],
            num_pairs: qmc.UInt,
        ) -> tuple[
            qmc.Vector[qmc.Bit],
            qmc.Vector[qmc.Bit],
            qmc.Bit,
        ]:
            # This allocation order keeps system on the least-significant wires.
            system = qmc.qubit_array(encoding.num_system_qubits, "system")
            signal = qmc.qubit_array(encoding.num_signal_qubits, "signal")
            projector_aux = qmc.qubit("projector_aux")
            system = qmc.amplitude_encoding(system, normalized_rhs)

            for pair in qmc.range(num_pairs):
                even_index = pair + pair
                odd_index = even_index + 1
                projector_aux, signal = projector_phase(
                    projector_aux,
                    signal,
                    phases[even_index] + phases[even_index],
                )
                signal, system = encoding.unitary(signal, system)
                projector_aux, signal = projector_phase(
                    projector_aux,
                    signal,
                    phases[odd_index] + phases[odd_index],
                )
                signal, system = inverse_unitary(signal, system)

            tail_index = num_pairs + num_pairs
            final_index = tail_index + 1
            projector_aux, signal = projector_phase(
                projector_aux,
                signal,
                phases[tail_index] + phases[tail_index],
            )
            signal, system = encoding.unitary(signal, system)
            projector_aux, signal = projector_phase(
                projector_aux,
                signal,
                phases[final_index] + phases[final_index],
            )
            return (
                qmc.measure(system),
                qmc.measure(signal),
                qmc.measure(projector_aux),
            )

    else:

        @qmc.qkernel
        def qsvt_kernel(
            phases: qmc.Vector[qmc.Float],
            num_pairs: qmc.UInt,
        ) -> tuple[
            qmc.Vector[qmc.Bit],
            qmc.Vector[qmc.Bit],
            qmc.Bit,
        ]:
            # This allocation order keeps system on the least-significant wires.
            system = qmc.qubit_array(encoding.num_system_qubits, "system")
            signal = qmc.qubit_array(encoding.num_signal_qubits, "signal")
            projector_aux = qmc.qubit("projector_aux")
            system = qmc.amplitude_encoding(system, normalized_rhs)

            for pair in qmc.range(num_pairs):
                even_index = pair + pair
                odd_index = even_index + 1
                projector_aux, signal = projector_phase(
                    projector_aux,
                    signal,
                    phases[even_index] + phases[even_index],
                )
                signal, system = encoding.unitary(signal, system)
                projector_aux, signal = projector_phase(
                    projector_aux,
                    signal,
                    phases[odd_index] + phases[odd_index],
                )
                signal, system = inverse_unitary(signal, system)

            final_index = num_pairs + num_pairs
            projector_aux, signal = projector_phase(
                projector_aux,
                signal,
                phases[final_index] + phases[final_index],
            )
            return (
                qmc.measure(system),
                qmc.measure(signal),
                qmc.measure(projector_aux),
            )

    return qsvt_kernel

## 6. inverse polynomialの位相生成

PR #549と同じくpyqspで $1/x$ の有界多項式近似を作り、QSVT規約へ
位相を変換します。pyqspが未導入の場合は、回路構造のtranspile確認だけを
行う3位相のsmoke sequenceへフォールバックします。

**重要:** smoke sequenceはPoisson逆演算の精度を示すものではありません。
数値精度を評価する場合はpyqspを導入し、生成された本来の位相列を使ってください。

In [ ]:
def convert_qsp_to_qsvt(phases_wx):
    """Convert pyqsp Wx phases to the QSVT convention used here."""
    phases_wx = np.asarray(phases_wx, dtype=float)
    degree = len(phases_wx) - 1
    phases = np.zeros_like(phases_wx)
    phases[0] = phases_wx[0] + (2 * degree - 1) * np.pi / 4
    for index in range(1, degree):
        phases[index] = phases_wx[index] - np.pi / 2
    phases[degree] = phases_wx[degree] - np.pi / 4
    return (phases + np.pi) % (2 * np.pi) - np.pi


def qsvt_inverse_phases(kappa=10, epsilon=0.1, tolerance=1e-5):
    """Generate inverse-polynomial QSVT phases or a smoke sequence."""
    try:
        from pyqsp import angle_sequence, poly as pyqsp_poly
    except (ModuleNotFoundError, ImportError):
        warnings.warn(
            "pyqsp is not installed; using a three-phase structural smoke "
            "sequence that does not demonstrate inverse accuracy.",
            stacklevel=2,
        )
        return np.array([np.pi / 4, 0.0, -np.pi / 4]), 1.0, True

    polynomial = pyqsp_poly.PolyOneOverX()
    coefficients, scale = polynomial.generate(
        kappa=kappa,
        epsilon=epsilon,
        return_coef=True,
        ensure_bounded=True,
        return_scale=True,
    )
    phases_wx = angle_sequence.QuantumSignalProcessingPhases(
        coefficients,
        signal_operator="Wx",
        tolerance=tolerance,
    )
    return convert_qsp_to_qsvt(phases_wx), scale, False


qsvt_phases, inverse_scale, using_smoke_phases = qsvt_inverse_phases(
    kappa=10,
    epsilon=0.1,
)
qsvt_bindings = {
    "phases": np.asarray(qsvt_phases, dtype=float).tolist(),
    "num_pairs": qsvt_num_pairs(len(qsvt_phases)),
}

print("QSVT phase count:", len(qsvt_phases))
print("Inverse-polynomial scale:", inverse_scale)
print("Structural smoke mode:", using_smoke_phases)

## 7. 1次元QSVTを実行する

Statevectorの先頭 $N$ 成分は、systemを最初にallocateしたため、
signalとprojector_auxを全て0へpostselectしたsystem成分です。

In [ ]:
qsvt_kernel_1d = make_qsvt_poisson_kernel(
    encoding_1d,
    normalized_rhs_1d,
    len(qsvt_phases),
)
qsvt_circuit_1d = transpiler.to_circuit(
    qsvt_kernel_1d,
    bindings=qsvt_bindings,
).remove_final_measurements(inplace=False)
state_1d = Statevector.from_instruction(qsvt_circuit_1d).data
postselected_1d = state_1d[:num_points_1d]
computed_1d = recover_scaled_solution(
    matrix_1d,
    rhs_1d,
    postselected_1d,
)

print("1D residual:", np.linalg.norm(matrix_1d @ computed_1d - rhs_1d))
print("1D fidelity:", fidelity(computed_1d, reference_1d))
print("1D maximum absolute error:", np.max(np.abs(computed_1d - reference_1d)))
print("1D Qiskit qubits:", qsvt_circuit_1d.num_qubits)
print("1D Qiskit depth:", qsvt_circuit_1d.depth())
if using_smoke_phases:
    print("The values above are a wiring smoke test, not an inverse-accuracy claim.")

## 8. 同じfactoryを2次元へ適用する

2次元で変わるのはcoefficientsとregister_sizesだけです。
QSVT factory本体はそのまま再利用します。各offset tupleの最初の成分は
flattened systemのLSB側axis、2番目は次のaxisへ作用します。

In [ ]:
num_points_2d = 4
matrix_2d, raw_rhs_2d, grid_x_2d, grid_y_2d, spacing_2d = build_2d_periodic_problem(
    num_points_2d,
    source=lambda x, y: np.sin(2.0 * np.pi * x) * np.sin(2.0 * np.pi * y),
)
rhs_2d = project_periodic_rhs(raw_rhs_2d)
reference_2d = np.linalg.pinv(matrix_2d) @ rhs_2d
normalized_rhs_2d = normalize_state(rhs_2d)
num_axis_qubits_2d = int(np.log2(num_points_2d))

encoding_2d = qmc.periodic_stencil_block_encoding(
    {
        (0, 0): -4.0 / spacing_2d**2,
        (-1, 0): 1.0 / spacing_2d**2,
        (1, 0): 1.0 / spacing_2d**2,
        (0, -1): 1.0 / spacing_2d**2,
        (0, 1): 1.0 / spacing_2d**2,
    },
    register_sizes=(num_axis_qubits_2d, num_axis_qubits_2d),
)

print("2D normalization:", encoding_2d.normalization)
print("2D signal qubits:", encoding_2d.num_signal_qubits)
print("2D system qubits:", encoding_2d.num_system_qubits)
print("2D canonical offsets:", encoding_2d.offsets)

In [ ]:
encoded_matrix_2d = extract_encoded_matrix(encoding_2d)
error_2d = np.max(np.abs(encoded_matrix_2d - matrix_2d))
print("2D block matches the finite-difference matrix:", error_2d < 1e-8)
print("2D maximum absolute error:", error_2d)
assert np.allclose(encoded_matrix_2d, matrix_2d, atol=1e-8)

In [ ]:
qsvt_kernel_2d = make_qsvt_poisson_kernel(
    encoding_2d,
    normalized_rhs_2d,
    len(qsvt_phases),
)
qsvt_circuit_2d = transpiler.to_circuit(
    qsvt_kernel_2d,
    bindings=qsvt_bindings,
).remove_final_measurements(inplace=False)
state_2d = Statevector.from_instruction(qsvt_circuit_2d).data
system_dimension_2d = num_points_2d * num_points_2d
postselected_2d = state_2d[:system_dimension_2d]
computed_2d = recover_scaled_solution(
    matrix_2d,
    rhs_2d,
    postselected_2d,
)

print("2D residual:", np.linalg.norm(matrix_2d @ computed_2d - rhs_2d))
print("2D fidelity:", fidelity(computed_2d, reference_2d))
print("2D maximum absolute error:", np.max(np.abs(computed_2d - reference_2d)))
print("2D Qiskit qubits:", qsvt_circuit_2d.num_qubits)
print("2D Qiskit depth:", qsvt_circuit_2d.depth())
if using_smoke_phases:
    print("The values above are a wiring smoke test, not an inverse-accuracy claim.")

## 9. 2次元結果の可視化

In [ ]:
reference_grid_2d = reference_2d.reshape(num_points_2d, num_points_2d)
computed_grid_2d = computed_2d.reshape(num_points_2d, num_points_2d)

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
reference_image = axes[0].imshow(
    reference_grid_2d,
    origin="lower",
    cmap="viridis",
)
axes[0].set_title("Classical pseudoinverse")
axes[0].set_xlabel("x index")
axes[0].set_ylabel("y index")
figure.colorbar(reference_image, ax=axes[0])

qsvt_image = axes[1].imshow(
    computed_grid_2d,
    origin="lower",
    cmap="plasma",
)
axes[1].set_title("QSVT postselected component")
axes[1].set_xlabel("x index")
axes[1].set_ylabel("y index")
figure.colorbar(qsvt_image, ax=axes[1])
figure.tight_layout()
plt.show()

## まとめ

今回のfactoryを使うと、PR #549で手書きされていた1次元・2次元それぞれの
PREPARE、制御shift、UNPREPAREを削除できます。QSVT側は次の共通契約だけを
利用します。

- unitary(signal, system) -> (signal, system)
- normalization
- num_signal_qubits
- num_system_qubits

その結果、1次元と2次元で別々だったQSVT factoryを1つへ統合できました。
encodingが内部で使うsignalと、QSVT projectorが一時的に使う補助qubitの
所有範囲も明確になります。

一方、このノートブックのPeriodic Stencil factoryは、power-of-twoサイズ、
周期境界、位置に依存しない係数を対象とします。非周期境界、変数係数、
runtimeで係数を変更するencodingには別の方式や別factoryが必要です。